# CloakUNet v5 — Pure Adversarial Training

## What changed vs v4
- **No MSE delta regression term** — was pulling U-Net toward near-zero scaled-down PGD deltas
- **Loss is purely**: `adv_loss = -mean((latent_cloak - latent_orig)²)` — maximize VAE latent distance
- **unet.py epsilon 16/255** (patched inline after clone) — double the perturbation budget
- **config.py LPIPS_THRESHOLD 0.10** — goal is AI disruption, minor visibility acceptable
- **Full resume-safe**: saves epoch, optimizer, scheduler, best_val_adv, patience_count every epoch
- **Drive upload verified** before training starts

## Why v4 was flat at -58
The MSE term `(delta_pred - delta_target)²` forced the U-Net to copy PGD deltas.
Those deltas were generated with LPIPS gate 0.05 which scaled them nearly to zero.
So the U-Net learned to predict near-zero deltas → no latent disruption.
Removing MSE entirely lets the U-Net find its own perturbation that maximizes disruption.

## Expected training behavior
- val_adv should move from ~0 toward -100 to -300 over 15-20 epochs
- If val_adv is still flat after epoch 3, something is wrong — check the print output
- visual_lpips will be 0.05-0.15 — acceptable given our goal

In [ ]:
# Cell 1: Install + Drive auth + verify upload works
import os
os.system('pip install -q PyDrive2 lpips diffusers transformers accelerate')

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

PAIRS_FOLDER_ID = '15-K1jg_MWqj6J0FJCDoaGm9aeKc0tgoC'
CKPT_FOLDER_ID  = '18cprgjVtQcEm4B9g-S9N0wpjHziGlUS5'

def upload_to_drive(local_path, folder_id, title=None):
    title = title or os.path.basename(local_path)
    try:
        existing = drive.ListFile(
            {'q': f"'{folder_id}' in parents and title='{title}' and trashed=false"}
        ).GetList()
        if existing:
            f = existing[0]; f.SetContentFile(local_path); f.Upload()
        else:
            f = drive.CreateFile({'title': title, 'parents': [{'id': folder_id}]})
            f.SetContentFile(local_path); f.Upload()
        print(f'  Drive up: {title}'); return True
    except Exception as e:
        print(f'  Upload failed {title}: {e}'); return False

# Verify upload works BEFORE wasting GPU time
with open('/tmp/test.txt', 'w') as f: f.write('test')
ok = upload_to_drive('/tmp/test.txt', CKPT_FOLDER_ID, '_auth_test.txt')
if not ok:
    raise RuntimeError('Drive upload failed — fix auth before continuing')
print('Drive auth OK')

In [7]:
# Cell 2: Clone repo + patch unet.py (16/255 eps) + patch config.py (LPIPS 0.10)
import subprocess, sys, torch
from pathlib import Path

REPO_PATH  = '/content/Luxe'
CKPT_LOCAL = '/content/checkpoints'
PAIRS_DIR  = '/content/unet_pairs'
os.makedirs(CKPT_LOCAL, exist_ok=True)
os.makedirs(PAIRS_DIR,  exist_ok=True)

if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', 'https://github.com/nabirakhan/luxe', REPO_PATH], check=True)
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull'], check=True)

# Patch 1: pgd_modification.py — swap ViT-L/14 → ViT-B/32 (VRAM workaround on T4)
mod_path = f'{REPO_PATH}/backend/pgd_modification.py'
with open(mod_path) as f: c = f.read()
c = c.replace(
    'clip_model, _ = clip.load("ViT-L/14", device=self._device)',
    'clip_model, _ = clip.load("ViT-B/32", device=self._device)'
)
with open(mod_path, 'w') as f: f.write(c)
print('pgd_modification.py patched: ViT-L/14 → ViT-B/32')

# Patch 2: unet.py — replace ALL occurrences of EPS_PGD with UNET_EPS = 16/255
# (docstring + import + forward line all get replaced)
unet_path = f'{REPO_PATH}/backend/unet.py'
with open(unet_path) as f: c = f.read()
c = c.replace(
    'from config import EPS_PGD',
    'UNET_EPS = 16 / 255  # doubled from EPS_PGD (8/255) for stronger disruption'
)
c = c.replace('EPS_PGD', 'UNET_EPS')   # catches docstring + forward line
with open(unet_path, 'w') as f: f.write(c)
print('unet.py patched: all EPS_PGD → UNET_EPS = 16/255')

# Patch 3: config.py — raise LPIPS_THRESHOLD 0.05 → 0.10
cfg_path = f'{REPO_PATH}/backend/config.py'
with open(cfg_path) as f: c = f.read()
c = c.replace('LPIPS_THRESHOLD          = 0.05', 'LPIPS_THRESHOLD          = 0.10')
with open(cfg_path, 'w') as f: f.write(c)
print('config.py patched: LPIPS_THRESHOLD 0.05 → 0.10')

sys.path.insert(0, f'{REPO_PATH}/backend')

# Verify patches
with open(unet_path) as f: unet_src = f.read()
assert 'UNET_EPS = 16 / 255' in unet_src,  'FAIL: UNET_EPS not in unet.py'
assert '* EPS_PGD' not in unet_src,         'FAIL: * EPS_PGD still in unet.py forward'
assert 'from config import EPS_PGD' not in unet_src, 'FAIL: old import still in unet.py'

with open(cfg_path) as f: cfg_src = f.read()
assert 'LPIPS_THRESHOLD          = 0.10' in cfg_src, 'FAIL: config.py patch failed'

print('All patches verified OK')

Already up to date.
pgd_modification.py patched: ViT-L/14 → ViT-B/32
unet.py patched: all EPS_PGD → UNET_EPS = 16/255
config.py patched: LPIPS_THRESHOLD 0.05 → 0.10
All patches verified OK


In [8]:
# Cell 3: Download training pairs — sequential (PyDrive2 is not thread-safe)
import time

MAX_DOWNLOAD = 2500

file_list = drive.ListFile(
    {'q': f"'{PAIRS_FOLDER_ID}' in parents and trashed=false"}
).GetList()

already  = {p.name for p in Path(PAIRS_DIR).glob('*.pt')}
to_get   = [f for f in file_list if f['title'] not in already]
to_get   = to_get[:max(0, MAX_DOWNLOAD - len(already))]

print(f'On Drive: {len(file_list)} | Local: {len(already)} | To download: {len(to_get)}')

if len(to_get) == 0:
    print('Nothing to download — all pairs already local')
else:
    ok, fail = 0, 0
    t_start  = time.time()

    for i, f in enumerate(to_get):
        dest = os.path.join(PAIRS_DIR, f['title'])
        tmp  = dest + '.tmp'

        # Skip if already complete
        if os.path.exists(dest) and os.path.getsize(dest) > 0:
            ok += 1
            continue

        try:
            f.GetContentFile(tmp)
            os.replace(tmp, dest)
            ok += 1
        except Exception as e:
            fail += 1
            if os.path.exists(tmp):
                os.remove(tmp)
            print(f'  FAIL [{i+1}] {f["title"]}: {e}')

        # Progress every 100 files
        if (i + 1) % 100 == 0:
            elapsed  = time.time() - t_start
            rate     = (i + 1) / elapsed
            eta_min  = (len(to_get) - i - 1) / rate / 60
            disk_mb  = sum(p.stat().st_size for p in Path(PAIRS_DIR).glob('*.pt')) / 1e6
            print(f'  [{i+1}/{len(to_get)}]  ok={ok}  fail={fail}  '
                  f'rate={rate:.1f}/s  ETA={eta_min:.0f}min  disk={disk_mb:.0f}MB')

    elapsed = time.time() - t_start
    print(f'\nDone: {ok} downloaded, {fail} failed in {elapsed/60:.1f} min')

# Validate — remove corrupted files
bad = []
for pt in Path(PAIRS_DIR).glob('*.pt'):
    try:
        torch.load(pt, map_location='cpu', weights_only=True)
    except Exception:
        bad.append(pt)
        pt.unlink()

if bad:
    print(f'Removed {len(bad)} corrupted files')

total = len(list(Path(PAIRS_DIR).glob('*.pt')))
print(f'{total} pairs ready in {PAIRS_DIR}')

On Drive: 2788 | Local: 2342 | To download: 158
  [100/158]  ok=100  fail=0  rate=1.4/s  ETA=1min  disk=17930MB

Done: 158 downloaded, 0 failed in 1.9 min
2500 pairs ready in /content/unet_pairs


In [9]:
# Cell 4: Download checkpoints
# For v5 we start fresh — do NOT load v4 weights (trained with conflicting MSE objective)
# Only need sd_inpaint_vae.pth for the frozen VAE

ckpt_files = drive.ListFile({'q': f"'{CKPT_FOLDER_ID}' in parents and trashed=false"}).GetList()
ckpt_names = {f['title']: f for f in ckpt_files}

print('Checkpoints on Drive:', list(ckpt_names.keys()))

# Always need the frozen VAE
NEEDED = ['sd_inpaint_vae.pth']

# Resume from v5 if it exists — NOT from v4 (different loss, incompatible optimizer state)
if 'unet_v5_resume.pth' in ckpt_names:
    NEEDED.append('unet_v5_resume.pth')
    print('Will resume from unet_v5_resume.pth')
else:
    print('No v5 resume found — starting from scratch (correct for first run)')

for fname in NEEDED:
    local = os.path.join(CKPT_LOCAL, fname)
    if os.path.exists(local):
        print(f'  Already local: {fname}'); continue
    if fname in ckpt_names:
        print(f'  Downloading {fname}...')
        ckpt_names[fname].GetContentFile(local)
        print(f'  Done: {fname} ({os.path.getsize(local)/1e6:.0f} MB)')
    else:
        print(f'  NOT ON DRIVE: {fname}')

print('\nCheckpoint status:')
for fname in ['sd_inpaint_vae.pth', 'unet_v5_resume.pth']:
    exists = os.path.exists(os.path.join(CKPT_LOCAL, fname))
    print(f'  {"OK" if exists else "MISSING"}: {fname}')

Checkpoints on Drive: ['_auth_test.txt', 'cloak_unet.pth', 'unet_resume.pth', 'sd_inpaint_vae.pth', 'ipp_checkpoints', 'segformer_lip.pth', 'segformer_checkpoints', 'inpaint_checkpoints', 'ipp_vae.pth', 'ip_adapter.pth']
No v5 resume found — starting from scratch (correct for first run)
  Done: sd_inpaint_vae.pth (335 MB)

Checkpoint status:
  OK: sd_inpaint_vae.pth
  MISSING: unet_v5_resume.pth


In [10]:
# Cell 5: Load frozen VAE (surrogate target)
from diffusers import AutoencoderKL
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

print('Loading VAE...')
vae = AutoencoderKL.from_pretrained(
    'runwayml/stable-diffusion-inpainting', subfolder='vae'
).to(DEVICE)

vae_ckpt = f'{CKPT_LOCAL}/sd_inpaint_vae.pth'
if os.path.exists(vae_ckpt):
    vae.load_state_dict(torch.load(vae_ckpt, map_location=DEVICE))
    print('Fine-tuned VAE weights loaded')
else:
    print('WARNING: sd_inpaint_vae.pth not found — using vanilla VAE')
    print('         Fine-tuned VAE = stronger white-box attack. Get it from Drive.')

vae.eval()
for p in vae.parameters():
    p.requires_grad_(False)
print('VAE frozen and ready')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
Loading VAE...


config.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

An error occurred while trying to fetch runwayml/stable-diffusion-inpainting: runwayml/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


vae/diffusion_pytorch_model.bin:   0%|          | 0.00/335M [00:00<?, ?B/s]

Fine-tuned VAE weights loaded
VAE frozen and ready


In [11]:
# Cell 6: Dataset
from torch.utils.data import Dataset, DataLoader, random_split

class PairDataset(Dataset):
    def __init__(self, folder):
        self.files = sorted(Path(folder).glob('*.pt'))
        if len(self.files) == 0:
            raise RuntimeError(f'No .pt files found in {folder}. Did Cell 3 run?')

    def __len__(self): return len(self.files)

    def __getitem__(self, i):
        s = torch.load(self.files[i], map_location='cpu', weights_only=True)
        # mask shape varies: [1, 512, 512] or [512, 512] — normalise to [1, 512, 512]
        mask = s['mask']
        if mask.dim() == 2: mask = mask.unsqueeze(0)
        # We no longer use delta_target in v5 — still return it for compat but ignored
        return s['x_orig'], s['delta'], mask

full_ds = PairDataset(PAIRS_DIR)
n_val   = max(1, int(len(full_ds) * 0.1))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                 generator=torch.Generator().manual_seed(42))
print(f'Dataset: {len(full_ds)} pairs | Train: {n_train} | Val: {n_val}')

Dataset: 2500 pairs | Train: 2250 | Val: 250


In [28]:
# Cell 7: Load U-Net + set hyperparameters + resume if v5 checkpoint exists
import time
from unet import CloakUNet

# ── Hyperparameters ──────────────────────────────────────────────────────────
EPOCHS     = 40       # more epochs since we're training from scratch with new loss
LR         = 1e-4
BATCH_SIZE = 4
PATIENCE   = 8        # stop if no improvement for 8 epochs

# No LAMBDA_MSE — pure adversarial loss

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)

unet      = CloakUNet().to(DEVICE)
optimiser = torch.optim.AdamW(unet.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda')

start_epoch    = 1
best_val_adv   = float('inf')   # val_adv is negative; 'inf' means no best yet
patience_count = 0

resume_path = f'{CKPT_LOCAL}/unet_v5_resume.pth'

if os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location=DEVICE)
    if isinstance(ckpt, dict) and 'model_state' in ckpt:
        unet.load_state_dict(ckpt['model_state'])
        optimiser.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        best_val_adv   = ckpt['best_val_adv']
        patience_count = ckpt['patience_count']
        start_epoch    = ckpt['epoch'] + 1
        print(f'Resumed from epoch {ckpt["epoch"]}  best_val_adv={best_val_adv:.3f}  patience={patience_count}/{PATIENCE}')
    else:
        # Legacy: weights-only checkpoint — load weights, start optimizer fresh
        unet.load_state_dict(ckpt)
        print('Loaded weights only (no optimizer state) — optimizer starts fresh')
else:
    print('No v5 resume found — training from scratch')

print(f'Starting from epoch {start_epoch}/{EPOCHS}  LR={LR}')

# Sanity check: confirm unet uses 16/255 eps
with torch.no_grad():
    dummy = torch.zeros(1, 3, 512, 512).to(DEVICE)
    out   = unet(dummy)
    max_val = out.abs().max().item()
print(f'U-Net output max abs value: {max_val:.4f}  (expected ≈ {16/255:.4f} = 16/255)')
assert max_val <= (16/255) + 1e-5, f'FAIL: output {max_val} exceeds 16/255 bound'
print('U-Net epsilon check passed')

Resumed from epoch 21  best_val_adv=-80.028  patience=2/8
Starting from epoch 22/40  LR=0.0001
U-Net output max abs value: 0.0627  (expected ≈ 0.0627 = 16/255)
U-Net epsilon check passed


In [29]:
# Cell 8: Training loop — pure adversarial loss, full resume-safe state saved every epoch
#
# Loss: adv_loss = -mean((latent_cloak - latent_orig)²)
# We MINIMIZE adv_loss = we MAXIMIZE latent distance = VAE gets confused
# No MSE regression term — U-Net finds its own perturbation strategy

print(f'Starting training — {len(train_loader)} steps/epoch')
print(f'Goal: val_adv should decrease (more negative) each epoch')
print(f'If flat after epoch 3, something is wrong\n')

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    unet.train()
    train_adv_total = 0.0

    for step, (x_orig, _delta_unused, _mask_unused) in enumerate(train_loader):
        x_orig = x_orig.to(DEVICE)  # [B, 3, 512, 512]

        with torch.amp.autocast('cuda'):
            delta_pred = unet(x_orig)                              # [B, 3, 512, 512]
            x_cloaked  = torch.clamp(x_orig + delta_pred, 0, 1)

            with torch.no_grad():
                latent_orig = vae.encode(
                    x_orig.float() * 2 - 1
                ).latent_dist.mean.detach()                        # [B, 4, 64, 64]

            latent_cloak = vae.encode(
                x_cloaked.float() * 2 - 1
            ).latent_dist.mean                                     # requires grad

            # ONLY loss: maximize squared latent distance
            adv_loss = -torch.mean((latent_cloak - latent_orig) ** 2)

        scaler.scale(adv_loss).backward()
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(unet.parameters(), max_norm=1.0)
        scaler.step(optimiser)
        scaler.update()
        optimiser.zero_grad()
        torch.cuda.empty_cache()

        train_adv_total += adv_loss.item()

        if (step + 1) % 50 == 0:
            print(f'  [ep{epoch} s{step+1}/{len(train_loader)}] '
                  f'adv={adv_loss.item():.3f}  '
                  f'latent_dist={-adv_loss.item():.3f}  '
                  f'mem={torch.cuda.memory_allocated()/1e9:.1f}GB')

    # ── Validation ───────────────────────────────────────────────────────────
    unet.eval()
    val_adv_total = 0.0
    with torch.no_grad():
        for x_orig, _, _ in val_loader:
            x_orig = x_orig.to(DEVICE)
            with torch.amp.autocast('cuda'):
                delta_pred   = unet(x_orig)
                x_cloaked    = torch.clamp(x_orig + delta_pred, 0, 1)
                latent_orig  = vae.encode(x_orig.float() * 2 - 1).latent_dist.mean
                latent_cloak = vae.encode(x_cloaked.float() * 2 - 1).latent_dist.mean
                val_adv_total += -torch.mean((latent_cloak - latent_orig) ** 2).item()
            torch.cuda.empty_cache()

    train_adv = train_adv_total / len(train_loader)
    val_adv   = val_adv_total   / len(val_loader)
    scheduler.step()

    print(f'\nEpoch {epoch:02d}/{EPOCHS}  '
          f'train_adv={train_adv:.3f} (latent_dist={-train_adv:.1f})  '
          f'val_adv={val_adv:.3f} (latent_dist={-val_adv:.1f})  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  '
          f'time={int(time.time()-t0)}s')

    # val_adv is negative — lower (more negative) = better
    if val_adv < best_val_adv:
        best_val_adv   = val_adv
        patience_count = 0
        torch.save(unet.state_dict(), f'{CKPT_LOCAL}/cloak_unet_v5.pth')
        # Also save as cloak_unet.pth — this is what protect.py looks for
        torch.save(unet.state_dict(), f'{CKPT_LOCAL}/cloak_unet.pth')
        upload_to_drive(f'{CKPT_LOCAL}/cloak_unet_v5.pth', CKPT_FOLDER_ID, 'cloak_unet_v5.pth')
        upload_to_drive(f'{CKPT_LOCAL}/cloak_unet.pth',    CKPT_FOLDER_ID, 'cloak_unet.pth')
        print(f'  New best val_adv={val_adv:.3f} (latent_dist={-val_adv:.1f}) — saved + uploaded')
    else:
        patience_count += 1
        print(f'  No improvement — patience {patience_count}/{PATIENCE}')

    # Full state checkpoint every epoch — true resume from here
    full_state = {
        'epoch':           epoch,
        'model_state':     unet.state_dict(),
        'optimizer_state': optimiser.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_val_adv':    best_val_adv,
        'patience_count':  patience_count,
    }
    torch.save(full_state, f'{CKPT_LOCAL}/unet_v5_resume.pth')
    upload_to_drive(f'{CKPT_LOCAL}/unet_v5_resume.pth', CKPT_FOLDER_ID, 'unet_v5_resume.pth')

    if patience_count >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

print(f'\nTraining complete. Best val_adv: {best_val_adv:.3f} (latent_dist={-best_val_adv:.1f})')
print('cloak_unet.pth and cloak_unet_v5.pth saved to Drive.')

Starting training — 563 steps/epoch
Goal: val_adv should decrease (more negative) each epoch
If flat after epoch 3, something is wrong



OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 25.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.30 GiB is allocated by PyTorch, and 95.22 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
scaler = torch.amp.GradScaler('cuda')

In [26]:
from google.colab import auth
from oauth2client.client import GoogleCredentials
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
print('Drive re-authed OK')

Drive re-authed OK


In [27]:
# Run this after any epoch where upload failed
torch.save(unet.state_dict(), f'{CKPT_LOCAL}/cloak_unet_v5.pth')
torch.save(unet.state_dict(), f'{CKPT_LOCAL}/cloak_unet.pth')
torch.save({
    'epoch':           epoch,
    'model_state':     unet.state_dict(),
    'optimizer_state': optimiser.state_dict(),
    'scheduler_state': scheduler.state_dict(),
    'best_val_adv':    best_val_adv,
    'patience_count':  patience_count,
}, f'{CKPT_LOCAL}/unet_v5_resume.pth')

upload_to_drive(f'{CKPT_LOCAL}/cloak_unet_v5.pth', CKPT_FOLDER_ID, 'cloak_unet_v5.pth')
upload_to_drive(f'{CKPT_LOCAL}/cloak_unet.pth',    CKPT_FOLDER_ID, 'cloak_unet.pth')
upload_to_drive(f'{CKPT_LOCAL}/unet_v5_resume.pth', CKPT_FOLDER_ID, 'unet_v5_resume.pth')
print('Manual save done')

  Drive up: cloak_unet_v5.pth
  Drive up: cloak_unet.pth
  Drive up: unet_v5_resume.pth
Manual save done


In [ ]:
# Cell 9: Disruption check after training
# Target: latent_dist > 80 AND visual_lpips < 0.15
# (Relaxed from old 0.05 target — we want disruption, minor visibility ok)
import lpips

unet_best = CloakUNet().to(DEVICE)
unet_best.load_state_dict(torch.load(f'{CKPT_LOCAL}/cloak_unet_v5.pth', map_location=DEVICE))
unet_best.eval()

loss_fn = lpips.LPIPS(net='alex').cpu()
all_val_files = [full_ds.files[i] for i in val_ds.indices]

print('Disruption check on 10 val images...')
print(f'{"image":<45} {"latent_dist":>12} {"visual_lpips":>13} {"status":>10}')
print('-' * 82)

latent_dists = []
visual_lpips_vals = []

for fp in all_val_files[:10]:
    s = torch.load(fp, map_location='cpu', weights_only=True)
    x = s['x_orig'].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            xc = torch.clamp(x + unet_best(x), 0, 1)
            lo = vae.encode(x.float()  * 2 - 1).latent_dist.mean
            lc = vae.encode(xc.float() * 2 - 1).latent_dist.mean
            ld = torch.mean((lc - lo) ** 2).item()

    vis = loss_fn(
        x.float().cpu()  * 2 - 1,
        xc.float().cpu() * 2 - 1
    ).item()

    latent_dists.append(ld)
    visual_lpips_vals.append(vis)

    ok_adv = 'GOOD' if ld  > 80   else ('PARTIAL' if ld  > 30  else 'WEAK')
    ok_vis = '  vis_ok' if vis < 0.15 else '  vis_high'
    print(f'{fp.name:<45} {ld:>12.2f} {vis:>13.4f} {ok_adv+ok_vis:>20}')

print('-' * 82)
avg_ld  = sum(latent_dists)      / len(latent_dists)
avg_vis = sum(visual_lpips_vals) / len(visual_lpips_vals)
print(f'{"AVERAGE":<45} {avg_ld:>12.2f} {avg_vis:>13.4f}')
print()
if avg_ld > 80 and avg_vis < 0.15:
    print('READY for full eval — latent_dist > 80 and visual_lpips < 0.15')
elif avg_ld > 30:
    print('PARTIAL — real disruption signal, but latent_dist < 80 target')
    print('Consider running more epochs from resume checkpoint')
else:
    print('WEAK — something wrong. Check training loss was decreasing in Cell 8')

In [ ]:
# Cell 10: Save cloak_unet.pth to Kaggle output tab (emergency backup)
import shutil

files_to_backup = [
    (f'{CKPT_LOCAL}/cloak_unet_v5.pth', '/content/cloak_unet_v5.pth'),
    (f'{CKPT_LOCAL}/cloak_unet.pth',    '/content/cloak_unet.pth'),
    (f'{CKPT_LOCAL}/unet_v5_resume.pth','/content/unet_v5_resume.pth'),
]

for src, dst in files_to_backup:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Backed up: {os.path.basename(dst)} ({os.path.getsize(dst)/1e6:.0f} MB)')
    else:
        print(f'NOT FOUND: {src}')

print()
print('To deploy: copy cloak_unet.pth to backend/checkpoints/ on your machine')
print('protect.py will automatically use it as the fast path')